# Infaction — City Placement LOCKED v0.4.6

Runs locked Faction/Zone/Location selection and Location→Zone placement; it deliberately **skips the obsolete v0.4.3 City experiment**; then applies **City Placement LOCKED** as an overlay. All selected Zones belong to one city; standalone Location can be Inside or Outside separately. Site generation is intentionally **not yet implemented**.

Sources: [City engine](https://github.com/playrenathea/rantau/blob/main/infaction_city_locked.py), [original City scores](https://github.com/playrenathea/rantau/blob/main/data/infaction_city_score_original_locked.json), [placement engine](https://github.com/playrenathea/rantau/blob/main/infaction_placement_locked.py), [master Drive specification](https://docs.google.com/document/d/15AVs2_mh66o-M8YnetGeLZqB2HuXRm04ZqIhNWWHT6s/edit).
**Canonical preflight:** single master JSON `data/infaction_master_v0.4.json` (original Book1 + approved overlays); the notebook asserts its embedded generator data match the master and reads the complete 118×118 affinity matrix. The separate original Book1 JSON is an archive/integrity cross-check, not another active source.


In [ ]:
import urllib.request, json, contextlib, io, types
ROOT='https://raw.githubusercontent.com/playrenathea/rantau/main/'
def get_text(path): return urllib.request.urlopen(ROOT+path,timeout=30).read().decode('utf-8')
source=json.loads(get_text('notebooks/Infaction_Faction_Naming_Zone_Location_Generator_v0.4.3_City_Experiment_Revised.ipynb'))
ns={'__name__':'__main__'}
for i in (1,2,3,5,8):
    with contextlib.redirect_stdout(io.StringIO()):
        exec(compile(''.join(source['cells'][i]['source']),str(i),'exec'),ns)
placement=types.ModuleType('infaction_placement_locked')
exec(compile(get_text('infaction_placement_locked.py'),'infaction_placement_locked.py','exec'),placement.__dict__)
city=types.ModuleType('infaction_city_locked')
exec(compile(get_text('infaction_city_locked.py'),'infaction_city_locked.py','exec'),city.__dict__)
neutral_data=json.loads(get_text('data/infaction_neutral_static_v2_locked.json'))
neutral={r['place']:r['neutral_affinity'] for r in neutral_data['records']}
city_scores=json.loads(get_text('data/infaction_city_score_original_locked.json'))
assert len(neutral)==len(city_scores)==110
print('Ready:',placement.VERSION,city.VERSION)

In [ ]:
# Mandatory canonical data preflight. Fail FAST before running any experiment.
MASTER=json.loads(get_text('data/infaction_master_v0.4.json'))
assert MASTER['selection_locked']=='0.4.2'
assert MASTER['source_file_sha256']=='a4d8a4c542b62f45445a18c9c781d87cc0e7638bc51bb65c1a3a6919edea0985'
assert len(MASTER['raw']['zone_selection']['zones'])==36
assert len(MASTER['raw']['location_selection']['locations'])==74
assert len(MASTER['overlay']['zone_affinity_overrides'])==164
assert MASTER['active']['zone_selection']['zones']==ns['ZONE_DATA']['zones'], 'Embedded Zone data drifted from master!'
assert MASTER['raw']['location_selection']['locations']==ns['LOCATION_DATA']['locations'], 'Embedded Location data drifted from master!'
assert city_scores=={p:int(v['inside_city_original']) for p,v in MASTER['raw']['place_catalog'].items() if v['scale'] in (1,2,4,5)}, 'City scores drifted from master!'
assert {**{p:int(v['inside_city_original']) for p,v in MASTER['raw']['place_catalog'].items() if v['scale'] in (4,5)}, **city.ZONE_CITY_OVERRIDES}==MASTER['overlay']['zone_city_current'], 'Zone City overlay drifted!'
pa=MASTER['raw']['book1_original']['Place Affinity']['rows']
place_names=[r[0] for r in pa[2:]]
assert len(place_names)==118 and len(set(place_names))==118
assert pa[1][113:231]==place_names, 'Place Affinity row/column labels do not match'
for i in range(118):
    row=pa[i+2][113:231]
    assert len(row)==118 and row[i]==100
    for j in range(118):
        value=row[j]
        assert isinstance(value,(int,float)) and 0<=value<=100 and value==pa[j+2][113+i]
scales={p:int(v['scale']) for p,v in MASTER['raw']['place_catalog'].items()}
assert len([p for p in place_names if scales.get(p) in (2,3,4)])==50
assert sum(scales.get(p)==2 for p in place_names)==27
assert sum(scales.get(p)==3 for p in place_names)==8
assert sum(scales.get(p)==4 for p in place_names)==15
SITE_AFFINITY={p:{q:pa[i+2][j+113] for j,q in enumerate(place_names)} for i,p in enumerate(place_names)}
print('PASS: 118x118 complete affinity; 50 Site types; 36 Zones; 74 Locations; 164 overrides; embedded generation matches master')

In [ ]:
SEED=8100000
world=ns['generate_world_with_locations'](SEED)
placed=placement.place_world(world,ns['ZONE_DATA'],ns['LOCATION_DATA'],neutral,seed=SEED)
result=city.city_place_world(placed,city_scores,seed=SEED)
print('Inside Zones:',result['inside_zones'])
print('Outside Zones:',result['outside_zones'])
print('Outside extra:',result['extra_outside_zone'])
print('Standalone Locations:',result['standalone_locations'])
print('All Locations city-tagged:',len(result['location_city']))
assert len(result['location_city'])==len(placed['locations'])

In [ ]:
from collections import Counter
N=100
stats=Counter()
for seed in range(8100000,8100000+N):
    w=ns['generate_world_with_locations'](seed)
    p=placement.place_world(w,ns['ZONE_DATA'],ns['LOCATION_DATA'],neutral,seed=seed)
    c=city.city_place_world(p,city_scores,seed=seed)
    stats.update({'worlds':1, 'outside_zones':c['outside_zone_count'],
        'standalone':len(c['standalone_locations']),
        'standalone_outside':sum(v['city']=='Outside' for v in c['standalone_locations'].values())})
    assert c['outside_zone_count']>=1
    assert set(c['standalone_locations'])==set(p['outside'])
    assert all(c['location_city'][loc]==c['zones'][v['zone']]['city'] for loc,v in p['placement'].items() if v['zone'] is not None)
    assert all(c['zones'][z]['city']=='Outside' for z in p['zones'] if c['zones'][z]['base_city_score']==0)
print('100-world smoke test:',dict(stats))

In [ ]:
from pathlib import Path
Path(f'infaction_world_city_{SEED}.json').write_text(json.dumps({'placement':placed,'city':result},ensure_ascii=False,indent=2))
print('City overlay exported; site grouping pending')